# Part 3 — Application: mean–variance frontier on ETF returns

Single-period long-only Markowitz on 11 ETFs (see `python/data/metadata.json`).

**Not investment advice.** Daily returns; sample estimates of mu and Sigma.

In [ ]:
%matplotlib inline

import json
import sys
from pathlib import Path

import polars as pl
from IPython.display import display, Markdown

REPO = Path.cwd().resolve()
if not (REPO / "scripts" / "run_all.py").exists():
    REPO = REPO.parents[1]
sys.path.insert(0, str(REPO / "python" / "src"))

from portfolio_linalg.config import load_config
from portfolio_linalg.covariance import build_covariance
from portfolio_linalg.fetch_data import fetch_and_cache, load_returns
from portfolio_linalg.frontier import compute_frontier, min_variance_portfolio
from portfolio_linalg.interpret import (
    asset_summary_table,
    frontier_summary_table,
    min_variance_weights_table,
)
from portfolio_linalg.plots import all_figures, generate_all

## 1. Data

In [ ]:
cfg = load_config(REPO / "python" / "config.yaml")
cache = cfg.data_dir / "returns.parquet"

if cache.exists():
    returns = load_returns(cfg)
    print(f"Loaded cached returns: {cache}")
else:
    print("No cache — fetching via httpx (Yahoo chart API)...")
    returns = fetch_and_cache(cfg)

meta = json.loads((cfg.data_dir / "metadata.json").read_text(encoding="utf-8"))
display(Markdown(f"**{meta['n_obs']}** daily rows, **{meta['n_assets']}** assets, "
                 f"{meta['date_min']} to {meta['date_max']} ({meta['source']})."))

## 2. Covariance and PSD check (`np.cov`, `np.linalg.eigh`)

In [ ]:
cov = build_covariance(returns)
display(Markdown(
    f"- PSD (min eigenvalue >= -1e-8): **{cov.is_psd}** (min λ = {cov.min_eigenvalue:.2e})\n"
    f"- Condition number κ(Σ) ≈ **{cov.condition_number:.1f}**"
))

## 3. Single-asset comparison (sample daily mu / sigma)

In [ ]:
display(asset_summary_table(cov))

## 4. CVXPY efficient frontier

In [ ]:
frontier = compute_frontier(cov, cfg)
mvp = min_variance_portfolio(cov, cfg)
fs = frontier_summary_table(cov, frontier)

display(Markdown("### Min-variance long-only portfolio"))
display(min_variance_weights_table(cov, cfg))
display(Markdown(
    f"Daily mu = **{mvp['mu']:.6f}**, sigma = **{mvp['sigma']:.6f}**"
))

display(Markdown("### Frontier endpoints (daily)"))
display(pl.DataFrame({
    "point": ["low return (≈ min-var)", "high return", "best mu/sigma on frontier"],
    "mu": [fs.mu_low, fs.mu_high, fs.best_mu],
    "sigma": [fs.sigma_low, fs.sigma_high, fs.best_sigma],
}))

display(Markdown("### Weights at high-return frontier point"))
display(fs.high_return_weights)

## 5. Figures (saved to `python/figures/` and shown inline)

In [ ]:
paths = generate_all(cov, frontier, cfg.figures_dir)
print("Saved:")
for p in paths:
    print(f"  {p}")

for fig in all_figures(cov, frontier):
    display(fig)

## 6. Interpretation

- **Minimum risk** in this sample is achieved mostly with low-volatility bond exposure (see min-variance weights), not the highest single-asset return.
- **Highest target return** on the frontier often concentrates in the best sample mean asset (frequently GLD in our window) — a known quirk of sample-based Markowitz.
- The frontier curve trades **daily** expected return for **daily** volatility; eigenvalues and correlations explain why diversification is limited across equity ETFs.
- Formal PSD / variance facts are in `lean/`; the QP solver is not verified in Lean.